In [7]:
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import numpy as np
import json

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
model_uri = "models:/fraud-detection@champion"
model = mlflow.sklearn.load_model(model_uri)

In [8]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import add_all_features

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [16]:
from model.preprocessor_pipe_evalueate import evaluate_model
X_test = add_all_features(test)
y_test = test['isFraud']
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("final_evaluation")
with mlflow.start_run(run_name="lightgbm_test_results"):
    metrics = evaluate_model(model, X_test, y_test)

    mlflow.log_param("model", "lightgbm")

    mlflow.log_param("number_of_features", X_test.shape[1])

    mlflow.log_metrics(metrics)

2026/08/28 22:37:08 INFO mlflow.tracking.fluent: Experiment with name 'final_evaluation' does not exist. Creating a new experiment.


🏃 View run lightgbm_test_results at: http://127.0.0.1:5000/#/experiments/7/runs/bf1afb82ae43494f89fc1cce2c001b8d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
